# Feature Engineering

This notebook documents how raw OHLCV data from `data/SPY_raw.csv`, `data/TSLA_raw.csv`, and `data/VIX_raw.csv` was transformed into the feature-engineered datasets used for modeling.

**Output files** (already present in `data/`):
- `data/SPY_features.csv` — 36 columns, ~2,515 rows
- `data/TSLA_features.csv` — 36 columns, ~2,515 rows

The original feature engineering code lives in `deprecated/Feature_engineering.ipynb`.
This notebook serves as a reference for understanding each feature's formula and purpose.

---

## Feature Reference

### Raw columns (from Yahoo Finance)
| Column | Description |
|---|---|
| `open` | Price at market open |
| `high` | Highest intraday price |
| `low` | Lowest intraday price |
| `close` | Unadjusted closing price |
| `adj_close` | Dividend-adjusted closing price (SPY differs from close; TSLA is identical) |
| `volume` | Shares traded |

### Derived features
| Feature | Formula | Purpose |
|---|---|---|
| `daily_return` | `close.pct_change(1)` | Today's % price change — captures short-term momentum |
| `weekly_return` | `close.pct_change(5)` | 5-day % price change — medium-term momentum |
| `ma_7` | `close.rolling(7).mean()` | 7-day moving average |
| `ma_21` | `close.rolling(21).mean()` | 21-day moving average |
| `ma_cross` | `1 if ma_7 > ma_21 else 0` | Golden/death cross signal — 1 when short MA is above long MA |
| `dist_from_ma21` | `(close - ma_21) / ma_21` | How far price is above/below its 21-day trend |
| `daily_range` | `(high - low) / close` | Intraday volatility normalized by close price |
| `rsi_14` | 14-day Wilder RSI | Overbought (>70) / oversold (<30) momentum oscillator |
| `macd` | `EMA(12) - EMA(26)` | MACD line — difference between short and long exponential moving averages |
| `macd_signal` | `EMA(macd, 9)` | Signal line — smoothed version of MACD |
| `macd_hist` | `macd - macd_signal` | Histogram — direction of MACD momentum |
| `bb_position` | `(close - ma_20) / (2 * std_20)` | Position within Bollinger Bands (-1 to +1) |
| `volatility_7` | `daily_return.rolling(7).std()` | Short-term realized volatility |
| `volatility_20` | `daily_return.rolling(20).std()` | Medium-term realized volatility |
| `volume_change` | `volume.pct_change(1)` | Day-over-day volume change |
| `volume_ma20` | `volume.rolling(20).mean()` | 20-day average volume |
| `volume_ratio` | `volume / volume_ma20` | Volume relative to recent average — captures unusual activity |
| `lag_return_1` | `daily_return.shift(1)` | Yesterday's return — tests short-term autocorrelation |
| `lag_return_3` | `daily_return.shift(3)` | 3-day lagged return |
| `lag_return_5` | `daily_return.shift(5)` | 5-day lagged return |
| `month` | `index.month` | Calendar month (1–12) |
| `quarter` | `index.quarter` | Calendar quarter (1–4) |
| `season_num` | 0=Winter, 1=Spring, 2=Summer, 3=Fall | Numeric season encoding |
| `season` | Spring/Summer/Fall/Winter | String season label |
| `is_earnings_week` | Binary flag | 1 if the week contains an earnings announcement |
| `vix` | VIX daily close | Merged from `VIX_raw.csv` by date |
| `is_major_event` | `1 if vix > 30 else 0` | Elevated market stress indicator |

### Targets
| Target | Formula | Task |
|---|---|---|
| `target_direction` | `1 if close.shift(-1) > close else 0` | Classification — next-day UP/DOWN |
| `target_return` | `close.pct_change(5).shift(-5)` | Regression — 5-day forward return |

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ── Load feature-engineered datasets ─────────────────────────────────────
spy  = pd.read_csv("data/SPY_features.csv",  parse_dates=["date"], index_col="date")
tsla = pd.read_csv("data/TSLA_features.csv", parse_dates=["date"], index_col="date")

print(f"SPY features:  {len(spy):,} rows | {spy.index[0].date()} \u2192 {spy.index[-1].date()}")
print(f"TSLA features: {len(tsla):,} rows | {tsla.index[0].date()} \u2192 {tsla.index[-1].date()}")
print(f"\nColumns ({len(spy.columns)}):")
print(list(spy.columns))

In [ ]:
# ── NaN analysis ─────────────────────────────────────────────────────────
# Rolling windows and lagged features produce NaN in early rows.
# Models drop these rows via dropna() before training.
print("SPY — NaN counts per column:")
nan_counts = spy.isnull().sum()
print(nan_counts[nan_counts > 0].to_string())

print(f"\nRows with any NaN: {spy.isnull().any(axis=1).sum()}")
print(f"Clean rows available for modeling: {spy.dropna().shape[0]:,}")

In [ ]:
# ── Feature summary statistics ────────────────────────────────────────────
MODEL_FEATURES = [
    "daily_return", "weekly_return", "lag_return_1", "lag_return_3", "lag_return_5",
    "dist_from_ma21", "ma_cross",
    "macd", "macd_signal", "macd_hist",
    "daily_range", "volatility_7", "volatility_20", "bb_position",
    "volume_change", "volume_ratio",
    "rsi_14",
    "is_major_event", "is_earnings_week",
]

print("SPY — Feature statistics (clean rows only):")
print(spy[MODEL_FEATURES].dropna().describe().round(4).to_string())

In [ ]:
# ── Target distribution ───────────────────────────────────────────────────
for ticker, df in [("SPY", spy), ("TSLA", tsla)]:
    clean = df.dropna(subset=["target_direction", "target_return"])
    up   = (clean["target_direction"] == 1).sum()
    down = (clean["target_direction"] == 0).sum()
    print(f"{ticker} target_direction: UP={up} ({up/len(clean)*100:.1f}%), DOWN={down} ({down/len(clean)*100:.1f}%)")
    print(f"{ticker} target_return:    mean={clean['target_return'].mean():.4f}  "
          f"std={clean['target_return'].std():.4f}  "
          f"min={clean['target_return'].min():.4f}  "
          f"max={clean['target_return'].max():.4f}")
    print(f"{ticker} is_major_event:   {int(clean['is_major_event'].sum())} days VIX > 30 "
          f"({clean['is_major_event'].mean()*100:.1f}%)")
    print()

In [ ]:
# ── Feature correlation matrix (top correlated pairs) ────────────────────
corr = spy[MODEL_FEATURES].dropna().corr().abs()

# Extract upper triangle pairs
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
pairs.columns = ["feature_a", "feature_b", "correlation"]
pairs = pairs.sort_values("correlation", ascending=False)

print("Top 15 correlated feature pairs (SPY):")
print(pairs.head(15).to_string(index=False))